# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nglfrsarthak/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
%pip -q install duckdb scikit-learn pandas

In [8]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token in a cell.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HFTOKEN1')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


In [9]:
import duckdb

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + HF_TOKEN + "')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
# Mid-panel month ONLY. _sample (June 2026, the natural outcome window) stays sealed.
MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print('pointed at:', REL + '/fact_content_daily_performance/month=2026-03/')

pointed at: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The contract, in plain words (5 answers):

1. **One row = one content item on one report date.** Grain: `report_date × client_hash_id × content_hash_id` in `fact_content_daily_performance` (Q1 proves it).
2. **Tables: one.** `fact_content_daily_performance`, partition `month=2026-03` only. No `dim_*` joins in this slice, no query table, no `_sample`.
3. **Time window + decision moment.** Features: 1–15 March 2026. Outcome: 16–31 March 2026. Decision moment: end of 2026-03-15 — everything the frame uses is knowable by then.
4. **What I predict (proxy label).** `is_declining_proxy = (imp_16_31 < 0.8 × imp_01_15)` on items with `imp_01_15 >= 100` — a measured, directional proxy for "lost >20% of impressions half-over-half", the same >20% rule the starter slice uses for `trend_direction == 'down'`.
5. **Deliberately excluded.** The `fact_content_query_90d` table and any `*_last30`-style window touching March 16+: its fixed 90-day window overlaps my outcome window, so using it would leak the future into the features. IDs (`client_hash_id`, `content_hash_id`) are join/group keys only, never features.

One row represents data-driven Growth prediction of a given search, product, or a website. It states the growth over a fixed weekly or monthly sessions.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields (all from the March partition) | Why here |
|---|---|---|
| Feature (×5, Mar 1–15 only) | `imp_first15`, `clk_first15`, `avg_pos_first15`, `days_seen_first15`, `ctr_first15` | Aggregates of `gsc_impressions` / `gsc_clicks` / `gsc_avg_position` over Mar 1–15 — knowable at the decision moment |
| Label / proxy | `is_declining_proxy` from `imp_last16 vs imp_first15` | The thing I rank by; computed from the outcome window, never a feature |
| Context | `report_date`, `client_hash_id`, `content_hash_id`, `ga4_data_available` | Grouping, windowing, availability filtering — never model inputs |
| Excluded | `fact_content_query_90d.*` columns; raw IDs as features; any Mar 16–31 aggregate as a feature | Query-table window overlaps the outcome window (leakage); IDs are pseudonyms with no signal; late-March aggregates are the future at decision time |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Exactly three verification queries (Q1–Q3). The feature frame and leak trap below are separate cells, not verification queries.

In [10]:
# Q1 — GRAIN: one row really is report_date x client x content.
# Zero rows back means the grain holds.
q1 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MONTH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print('duplicate-grain rows:', len(q1))
q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate-grain rows: 0


,report_date,client_hash_id,content_hash_id,c


In [11]:
# Q2 — SLICE row count + date span: is month=2026-03 what it claims to be?
q2 = con.sql(f"""
    SELECT COUNT(*) AS march_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {MONTH}
""").df()
q2

,march_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [12]:
# Q3 — AVAILABILITY: ga4_data_available is THREE-valued (TRUE/FALSE/NULL).
# `= TRUE` would silently drop the NULLs, so filter with IS TRUE and show survivors.
q3 = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_ok_rows,
           SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_false_rows,
           SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS ga4_null_rows,
           ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_surviving_is_true
    FROM {MONTH}
""").df()
q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_ok_rows,ga4_false_rows,ga4_null_rows,pct_surviving_is_true
0,9841378,413966.0,6408671.0,3018741.0,4.21


### Feature frame (5 features, decision moment 2026-03-15)

One row per content item, GSC-side only (so the `IS TRUE` availability question above cannot bite the frame). Volume floor `imp_first15 >= 100` mirrors the worked notebook's `HAVING` guard.

In [13]:
frame = con.sql(f"""
    SELECT content_hash_id,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first15,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_first15,
           AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END) AS avg_pos_first15,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN 1 ELSE 0 END) AS days_seen_first15,
           SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_last16
    FROM {MONTH}
    GROUP BY 1
    HAVING imp_first15 >= 100
""").df()

frame['ctr_first15'] = frame['clk_first15'] / frame['imp_first15']
# Honest proxy label: >20% impression drop, second half vs first half.
frame['is_declining_proxy'] = (frame['imp_last16'] < 0.8 * frame['imp_first15']).astype(int)

print(f'content items with enough history: {len(frame):,}')
print(f"measured decline rate (decision-support, not a claim): {frame['is_declining_proxy'].mean():.3f}")
frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

content items with enough history: 77,540
measured decline rate (decision-support, not a claim): 0.285


,content_hash_id,imp_first15,clk_first15,avg_pos_first15,days_seen_first15,imp_last16,ctr_first15,is_declining_proxy
0,content_d0dff76c889de68f,111.0,0.0,5.222776,13.0,70.0,0.000000,1
1,content_2e6360ad20fd7107,219.0,1.0,3.737399,15.0,680.0,0.004566,0
2,content_65c50dfe9d87a585,1494.0,0.0,6.156643,14.0,1614.0,0.000000,0
3,content_d49a012dcb924e31,246.0,0.0,4.520919,15.0,83.0,0.000000,1
4,content_614baf2af4330bd7,413.0,1.0,4.390322,15.0,359.0,0.002421,0


Available-when lines (one per feature):

1. `imp_first15` — knowable at the decision moment because it sums only Mar 1–15 rows.
2. `clk_first15` — knowable at the decision moment because it sums only Mar 1–15 rows.
3. `avg_pos_first15` — knowable at the decision moment because it averages only Mar 1–15 positions.
4. `days_seen_first15` — knowable at the decision moment because it counts only Mar 1–15 days with impressions.
5. `ctr_first15` — knowable at the decision moment because it divides two Mar 1–15 aggregates.

In [14]:
# THE TRAP (leakage lesson, notebook-02 style, on real warehouse data):
# honest score -> add ONE label-derived column -> watch it jump -> delete it.
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

HONEST = ['imp_first15', 'clk_first15', 'avg_pos_first15', 'days_seen_first15', 'ctr_first15']
Xh = frame[HONEST].fillna(0.0)
y = frame['is_declining_proxy']
Xtr, Xte, ytr, yte = train_test_split(Xh, y, test_size=0.25, random_state=42, stratify=y)

floor = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr).score(Xte, yte)
honest = LogisticRegression(max_iter=1000).fit(Xtr, ytr).score(Xte, yte)
print(f'floor (majority class): {floor:.3f}')
print(f'HONEST accuracy (5 pre-decision features): {honest:.3f}')

# Spring the trap: imp_last16 is computed from the outcome window — it IS the label's raw material.
Xl = frame[HONEST + ['imp_last16']].fillna(0.0)
Xtr2, Xte2, ytr2, yte2 = train_test_split(Xl, y, test_size=0.25, random_state=42, stratify=y)
leaked = LogisticRegression(max_iter=1000).fit(Xtr2, ytr2).score(Xte2, yte2)
print(f'LEAKED accuracy (+ imp_last16): {leaked:.3f}  <- the jump toward perfect; this number is a lie')

# Delete it and keep the honest number.
del Xl, Xtr2, Xte2, ytr2, yte2
KEPT_ACCURACY = honest
print(f'KEPT accuracy (leak removed): {KEPT_ACCURACY:.3f}')

floor (majority class): 0.715
HONEST accuracy (5 pre-decision features): 0.716
LEAKED accuracy (+ imp_last16): 1.000  <- the jump toward perfect; this number is a lie
KEPT accuracy (leak removed): 0.716


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: unbalanced-panel coverage bias.** Per-client history depth differs (check `dim_clients.gsc_data_start`), so a fixed March-2026 slice over-represents long-history clients and silently drops young clients — my measured decline rate describes *March survivors with ≥100 early impressions*, directionally useful for triage, never a claim about all content. (Second limit, noted not modeled: rows with `ga4_data_available IS NOT TRUE` carry zero-filled/NULL engagement — I kept this frame GSC-only on purpose.)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.